# **Notebook d'Interprétabilité BERT/RoBERTa pour NLI**
Ce notebook automatise l'analyse des décisions de votre modèle en utilisant les **Integrated Gradients** et les **Matrices d'Attention**. — version GitHub propre

In [ ]:
# Installation des bibliothèques nécessaires
!pip install -q transformers[torch] datasets evaluate accelerate optuna scikit-learn matplotlib pandas seaborn captum

# Vérification du GPU
import torch
print(f"GPU détecté : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN GPU'}")

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers_interpret import SequenceClassificationExplainer

# Configuration de l'affichage
%matplotlib inline
sns.set_theme(style="white")

## 2. Chargement du modèle et du tokenizer
Modifiez le chemin `MODEL_PATH` pour pointer vers votre sauvegarde sur Google Drive.

In [ ]:
# Chemin local du modèle fine-tuné à interpréter.
# Modifiez cette variable selon le modèle que vous souhaitez analyser.
MODEL_PATH = "./models/final_bert_snli_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    output_attentions=True,
)
model.eval()

print("Modèle chargé depuis :", MODEL_PATH)

## 3. Fonctions de Visualisation
Nous créons une fonction pour générer des Heatmaps d'attention entre la Prémisse et l'Hypothèse.

In [ ]:
def plot_attention_heatmap(premise, hypothesis, layer=11, head=None):
    """
    Génère une heatmap montrant comment les mots de l'hypothèse regardent la prémisse.
    layer: index de la couche (0 à 11 pour BERT-base)
    head: index de la tête (0 à 11). Si None, on moyenne toutes les têtes.
    """
    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True, padding=True)
    if torch.cuda.is_available():
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        # attentions shape: (num_layers, batch, num_heads, sequence_length, sequence_length)
        attentions = outputs.attentions

    # Récupération des tokens pour les axes
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    # Extraction de la couche spécifiée
    attn_matrix = attentions[layer][0] # Shape (num_heads, seq, seq)

    if head is not None:
        attn_matrix = attn_matrix[head]
    else:
        attn_matrix = attn_matrix.mean(dim=0) # Moyenne des têtes

    attn_matrix = attn_matrix.cpu().numpy()

    # Affichage avec Seaborn
    plt.figure(figsize=(10, 8))
    sns.heatmap(attn_matrix, xticklabels=tokens, yticklabels=tokens, cmap="viridis", annot=False)
    plt.title(f"Attention Layer {layer} (Moyenne des têtes)" if head is None else f"Attention Layer {layer} Head {head}")
    plt.xlabel("Key (Ce qui est regardé)")
    plt.ylabel("Query (Ce qui regarde)")
    plt.show()

## 4. Analyse des Gradients (Importance des mots)
Cette partie utilise les Integrated Gradients pour voir quels mots ont causé la prédiction.

In [ ]:
def analyser_importance(premise, hypothesis):
    """
    Affiche l'importance de chaque mot pour la classe prédite.
    """
    word_attributions = cls_explainer(premise, hypothesis)

    # Affichage visuel interactif HTML dans le notebook
    cls_explainer.visualize()

    return word_attributions

## 5. Pipeline Automatisé
Utilisez cette fonction pour tester n'importe quelle paire de phrases et extraire toutes les données d'un coup.

In [ ]:
def pipeline_interpretabilite(p, h):
    print(f"--- ANALYSE NLI ---")
    print(f"Prémisse : {p}")
    print(f"Hypothèse : {h}")

    # 1. Prédiction brute
    inputs = tokenizer(p, h, return_tensors="pt")
    if torch.cuda.is_available(): inputs = {k:v.to("cuda") for k,v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
        pred_idx = torch.argmax(logits, dim=1).item()
        # Attention: Vérifiez l'ordre des labels de votre modèle (0=Entailment, 1=Neutral, 2=Contradiction pour SNLI généralement)
        labels = {0: "Entailment", 1: "Neutral", 2: "Contradiction"}
        print(f"\n🤖 Prédiction du modèle : {labels.get(pred_idx, pred_idx)}")

    print("\n🔍 1. Importance des mots (Gradients) :")
    analyser_importance(p, h)

    print("\n🗺️ 2. Carte d'Attention (Interaction spatiale) :")
    # On regarde souvent la dernière couche (layer=11) car c'est celle qui consolide la décision
    plot_attention_heatmap(p, h, layer=11)

# --- TEST AUTOMATIQUE ---
p_test = "Un chat noir dort sur le canapé."
h_test = "Un animal se repose dans le salon."
pipeline_interpretabilite(p_test, h_test)